## Laborator 4 - Data Normalization and SVM

In [83]:
from sklearn import preprocessing
import numpy as np

x_train = np.array([[1, -1, 2], [2, 0, 0], [0, 1, -1]], dtype=np.float64)
x_test = np.array([[-1, 1, 0]], dtype=np.float64)

# facem statisticile pe datele de antrenare
scaler = preprocessing.StandardScaler()
scaler.fit(x_train)
# afisam media
print(scaler.mean_) # => [1. 0. 0.33333333]
# afisam deviatia standard
print(scaler.scale_) # => [0.81649658 0.81649658 1.24721913]

# scalam datele de antrenare
scaled_x_train = scaler.transform(x_train)
print(scaled_x_train) # => [[0. -1.22474487 1.33630621]
# [1.22474487 0. -0.26726124]
# [-1.22474487 1.22474487 -1.06904497]]

# scalam datele de test
scaled_x_test = scaler.transform(x_test)
print(scaled_x_test) # => [[-2.44948974 1.22474487 -0.26726124]]

[1.         0.         0.33333333]
[0.81649658 0.81649658 1.24721913]
[[ 0.         -1.22474487  1.33630621]
 [ 1.22474487  0.         -0.26726124]
 [-1.22474487  1.22474487 -1.06904497]]
[[-2.44948974  1.22474487 -0.26726124]]


1. Descarcati si cititi datele de train si de test (atentie la formatul mesajelor)

In [85]:
# Citim datele de train si test
training_sentences = np.load("data/training_sentences.npy", allow_pickle=True)
test_sentences = np.load("data/test_sentences.npy", allow_pickle=True)
train_labels = np.load("data/training_labels.npy", allow_pickle=True)
test_labels = np.load("data/test_labels.npy", allow_pickle=True)

2. Definiți funcția normalize_data(train_data, test_data, type=None) care primește ca
parametri datele de antrenare, respectiv de testare și tipul de normalizare ({None,
‘standard’, ‘l1’, ‘l2’}) și întoarce aceste date normalizate.


In [84]:
from sklearn.preprocessing import StandardScaler, Normalizer

def normalize_data(train_data, test_data, type=None):
    if type is None:
        return train_data, test_data

    if type == 'standard':
        scaler = StandardScaler()
        train_data = scaler.fit_transform(train_data)
        test_data = scaler.transform(test_data)
    elif type == 'l1':
        normalizer = Normalizer(norm='l1')
        train_data = normalizer.fit_transform(train_data)
        test_data = normalizer.transform(test_data)
    elif type == 'l2':
        normalizer = Normalizer(norm='l2')
        train_data = normalizer.fit_transform(train_data)
        test_data = normalizer.transform(test_data)

    return train_data, test_data

3. Definiți clasa BagOfWords în al cărui constructor se inițializează vocabularul (un
dicționar gol). În cadrul ei implementați metoda build_vocabulary(self, data) care
primește ca parametru o listă de mesaje(listă de liste de strings) și construiește
vocabularul pe baza acesteia. Cheile dicționarului sunt reprezentate de cuvintele din
eseuri, iar valorile de id-urile unice atribuite cuvintelor. Pe lângă vocabularul pe care-l
construiți, rețineți și o listă cu cuvintele în ordinea adăugării în vocabular.
Afișați dimensiunea vocabularul construit (9522).
OBS. Vocabularul va fi construit doar pe baza datelor din setul de antrenare.


4. Definiți metoda get_features(self, data) care primește ca parametru o listă de
mesaje de dimensiune num_samples(listă de liste de strings) și returnează o matrice
de dimensiune (num_samples x dictionary_length) definită astfel:
features(sample_idx, word_idx) = numarul de aparitii al
cuvantului cu id − ul word_idx in documentul sample_idx

In [86]:
import numpy as np

class BagOfWords:
    def __init__(self):
        self.vocabulary = {}             # Dictionar {word: id}
        self.word_list = []              # Lista in ordinea adăugarii

    def build_vocabulary(self, data):
        word_id = 0
        for message in data:
            for word in message:
                if word not in self.vocabulary:
                    self.vocabulary[word] = word_id
                    self.word_list.append(word)
                    word_id += 1
    
    def get_features(self, data):
        # All the features (the 2D matrix of dimension no_of_messages x dictionary_vocab_length)
        feature_matrix = []
        for message in data:
            # Features per work
            frequency = [0] * len(self.word_list)
            for word in message:
                if word in self.vocabulary:
                    frequency[self.vocabulary[word]] += 1
            feature_matrix.append(frequency)
        return np.array(feature_matrix)

In [87]:
bow = BagOfWords()
bow.build_vocabulary(training_sentences)
print(len(bow.vocabulary))

9522


5. Cu ajutorul funcțiilor definite anterior, obțineți reprezentările BOW pentru mulțimea de
antrenare și testare, apoi normalizați-le folosind norma “L2”.


In [88]:
train_features = bow.get_features(training_sentences)
test_features = bow.get_features(test_sentences)
train_data, test_data = normalize_data(train_features, test_features, type = 'l2')

6. 
a) Antrenați un SVM cu kernel linear care să clasifice mesaje în mesaje
spam/non-spam. Pentru parametrul C setați valoarea 1.

In [99]:
from sklearn.svm import LinearSVC

# Antrenam un SVM liniar
linear_svm = LinearSVC(C=1)
linear_svm.fit(train_features, train_labels)

LinearSVC(C=1)

In [100]:
predicted_labels_test = linear_svm.predict(test_features) # Predict

In [101]:
# Accuracy score
# svm_model.score(test_features, test_labels)

b) Calculați acuratețea și
F1-score pentru mulțimea de testare.


In [102]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(test_labels, predicted_labels_test)
f1 = f1_score(test_labels, predicted_labels_test)

print("Acuratețea:", accuracy)
print("F1-score:", f1)

Acuratețea: 0.9853260869565217
F1-score: 0.9443298969072165


c) Afișați cele mai negative (spam) 10 cuvinte și cele mai pozitive (non-spam) 10
cuvinte.
the first 10 negative words are ['Text' 'To' 'mobile' 'CALL' 'FREE' 'txt' '&' 'Call' 'Txt'
'STOP']
the first 10 positive words are ['&lt#&gt' 'me' 'i' 'Going' 'him' 'Ok' 'I' 'Ill' 'my' 'Im']

In [94]:
# Luam coeficientii
coef = linear_svm.coef_[0]  # pentru clasificare binara

# Legam coeficientii de cuvintele lor
feature_weights = list(zip(bow.word_list, coef))

# Sortam dupa importanta
sorted_by_weight = sorted(feature_weights, key=lambda x: x[1])

#f(x) > 0 → modelul prezice clasa 1 (spam)
#f(x) < 0 → modelul prezice clasa 0 (non-spam).
top_spam_words = [word for word, weight in sorted_by_weight[-10:]]
top_nonspam_words = [word for word, weight in sorted_by_weight[:10]]

print("Top 10 most frequent spam words in test set:", top_spam_words)
print("Top 10 most frequent non-spam words in test set:", top_nonspam_words)

Top 10 most frequent spam words in test set: ['For', 'httptms', 'widelivecomindex', 'Txt', '85233', 'FREE>RingtoneReply', 'won', 'REAL', 'ringtoneking', '84484']
Top 10 most frequent non-spam words in test set: ['&lt#&gt', 'him', 'Oh', 'Alright', 'me', 'always', 'right', 'It', 'Ill', 'Waiting']
